# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [9]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [10]:
# Initialize and constants

# load_dotenv(override=True)
# api_key = os.getenv('OPENAI_API_KEY')

# if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
#     print("API key looks good so far")
# else:
#     print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

MODEL = 'gemma4:e4b-mlx'
OLLAMA_BASE_URL = "http://localhost:11434/v1"
from openai import OpenAI
openai = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [11]:
def parse_json_response(text):
    # gemma4:e4b-mlx wraps JSON replies in ```json fences even with response_format set, so scan for where valid JSON actually starts
    decoder = json.JSONDecoder()
    for i, ch in enumerate(text):
        if ch == "{":
            try:
                obj, _ = decoder.raw_decode(text, i)
                return obj
            except json.JSONDecodeError:
                continue
    raise ValueError(f"No valid JSON object found in: {text!r}")

In [12]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [13]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://company.com/about"},
        {"type": "careers page", "url": "https://company.com/careers"}
    ]
}
"""

In [14]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company,
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [15]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company,
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/

In [16]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    # print(result)
    links = parse_json_response(result)
    return links

In [17]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page', 'url': 'https://edwarddonner.com'},
  {'type': 'product/service page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'service offerings page',
   'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'company/leadership profile',
   'url': 'https://www.linkedin.com/in/eddonner/'}]}

In [18]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = parse_json_response(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [19]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gemma4:e4b-mlx
Found 3 relevant links


{'links': [{'type': 'company page', 'url': 'https://edwarddonner.com'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company website', 'url': 'https://nebula.io/'}]}

In [20]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemma4:e4b-mlx
Found 5 relevant links


{'links': [{'type': 'careers page',
   'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise solution', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'company profile',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'company blog', 'url': 'https://huggingface.co/blog'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [21]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        try:
            result += f"\n\n### Link: {link['type']}\n"
            result += fetch_website_contents(link["url"])
        except Exception as e:
            print(f"Failed to fetch {link['url']}: {e}")
    return result

In [22]:
print(fetch_page_and_all_relevant_links("https://edwarddonner.com"))

Selecting relevant links for https://edwarddonner.com by calling gemma4:e4b-mlx
Found 5 relevant links
## Landing Page:

Home - Edward Donner

Skip to content
Avatar
Curriculum
Proficiency
C4
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of AI startup
Nebula.io
. I was previously founder and CEO of AI startup untapt,
acquired in 2021
, and a Managing Director at JPMorgan.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 900,000 enrollme

In [23]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [24]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [25]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemma4:e4b-mlx
Found 3 relevant links


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nQwen/Qwen3.8-27B\nUpdated\n10 days ago\n•\n2.36M\n•\n12.4k\nunsloth/Qwen3.8-27B-GGUF\nUpdated\n4 days ago\n•\n6.67M\

In [26]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [27]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemma4:e4b-mlx
Found 6 relevant links


# 🌐 Hugging Face: Building the Future of Open AI

**The Home of Machine Learning. The Collaboration Platform for the Global AI Community.**

At Hugging Face, we are not just building AI models—we are building the ecosystem that powers the next generation of intelligence. We provide the central hub where researchers, scientists, engineers, and businesses can share, discover, experiment, and collaboratively build a more open, ethical, and powerful AI future. We are at the heart of the AI revolution.

***

## 🚀 For Our Customers & Partners (Solutions)

Whether you are a cutting-edge research lab or a global enterprise, Hugging Face provides the tools to turn complex machine learning theory into deployable reality.

**Our Platform Offers:**

*   **Models:** Access to a massive catalog of community-driven and optimized models (2M+). Explore state-of-the-art advancements, from efficient runtimes to advanced generative capabilities.
*   **Datasets:** A vast repository of diverse, curated data (500k+), allowing researchers to rapidly train and fine-tune models with high-quality data.
*   **Spaces & Apps:** Deploy and experiment with functional AI applications in real-time. Our platform hosts living demos, allowing you to test concepts, from automated music generation to complex image editing, before production deployment.
*   **Enterprise Solutions:** For organizations requiring reliability and scale, we offer robust **Enterprise Support**, **Inference Endpoints**, and **Storage Buckets**. Transform your ML pipelines from experimental projects into stable, powerful business tools.
*   **HuggingChat:** Utilize our sophisticated AI interface for advanced conversation and task completion, powered by the best models available.

***

## 💡 For Investors (Vision & Market Position)

We are positioned not just as a technology provider, but as a fundamental infrastructure layer in the accelerating global AI market.

**Why Invest in Hugging Face?**

*   **Ecosystem Lock-in:** We have successfully transitioned from a technical resource into a mission-critical platform that drives data, models, and applications simultaneously. The growth of our user base ensures the growth of our platform.
*   **The Open AI Movement:** Our core mission is centered on fostering open, ethical AI. This aligns perfectly with the increasing market demand for transparent, reproducible, and auditable AI solutions—a major differentiator in the current trajectory of the industry.
*   **Community Power:** Our strength is the speed and scale of the global machine learning community we support. We are a nexus of innovation, enabling a rapid feedback loop between scientific discovery and commercial application.
*   **Diversified Revenue Streams:** We serve the individual explorer, the academic researcher, and the Fortune 500 enterprise, providing varied revenue paths through Pro subscriptions, enterprise services, and specialized inference offerings.

***

## 🌟 For Potential Recruits (Culture & Careers)

If you are passionate about the intersection of technology, science, and global impact, this is where your work belongs.

**The Hugging Face Experience:**

*   **A Mission-Driven Environment:** We are driven by a powerful, shared goal: to help "build an open and ethical AI future together." Our team is dedicated to tackling the biggest challenges in modern AI.
*   **Cutting-Edge Focus:** We are backed by a "talented science team exploring the edge of tech." You will be working on problems that are defining the next era of computing.
*   **The Community Spirit:** Hugging Face is defined by collaboration. We are a fast-growing community that values shared knowledge and open contribution, creating a dynamic, high-energy work environment.
*   **Grow With Us:** Ready to build something that impacts millions? Join a company at the heart of the AI explosion, where your contributions directly shape the direction of the industry.

**Join the AI Revolution. Define the Future.**

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [28]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [29]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemma4:e4b-mlx
Found 5 relevant links


# 🤖 Hugging Face: Building the Future of AI, Together

## A Platform, A Community, A Movement.

Hugging Face is not just a company; we are the central ecosystem where the world's machine learning community collaborates, discovers, and deploys the next generation of AI. We are dedicated to solving and democratizing artificial intelligence, ensuring that cutting-edge technology is open, accessible, and ethical for everyone.

---

### 🌐 For Our Customers & Partners (The Platform Value)

Whether you are an independent researcher, a startup, or a major enterprise, Hugging Face provides the foundational tools needed to build, share, and run powerful AI applications. Our ecosystem is the home for ML brilliance:

*   **📚 The Hugging Face Hub:** Access an unparalleled repository of open-source resources. Users explore **2M+ Models**, **500k+ Datasets**, and **1M+ running Applications (Spaces)** daily.
*   **🛠️ Complete Tooling:** Beyond raw assets, we offer robust enterprise solutions, including dedicated **Inference Providers**, **Inference Endpoints**, and secure **Storage Buckets**, allowing institutions to deploy complex models reliably.
*   **🧠 AI in Action:** From advanced Natural Language Processing (NLP) to Deep Learning and image editing, our tools help you transform raw data into live, functional AI services.

**Our Promise:** We enable reproducible science and collaborative development by hosting the tools, not just the data.

---

### ✨ Culture & Vision (For Investors & Recruits)

At Hugging Face, our culture is built on the principles of openness, rigorous science, and collective innovation. We are a dynamic, internationally focused team dedicated to pushing the boundaries of technology while committing to ethical AI practices.

**Meet the Company DNA:**
*   **Open & Collaborative:** We are fundamentally a community platform. Our success is measured by the open-source tools and the shared knowledge we empower.
*   **Forward-Thinking:** We are at the heart of the AI revolution, driven by a talented group of scientists and engineers who are constantly exploring the edge of what is possible.
*   **Mission-Driven:** Our core mission is to democratize technology. We believe that the future of AI must be built collaboratively and responsibly.

---

### 🚀 Join the Revolution (Careers)

If you are a passionate scientist, engineer, or innovator focused on solving the grand challenges of machine learning, Hugging Face is the place for you. We pride ourselves on a vibrant, challenging, and collaborative environment.

**What We Offer:**
*   The chance to work on technologies that are fundamentally changing how the world interacts with intelligence.
*   A culture that champions learning and contribution.
*   Opportunity to be part of a global, diverse team dedicated to open science.

**Ready to contribute?**
Explore our current openings and see how you can help us build a more open and ethical AI future.

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>